In [ ]:
#created on 17/04/2026 by James McLoughlin

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning) # silences future development warninigs

import earthaccess
import folium
import geopandas as gpd

import numpy as np
np.seterr(divide='ignore', invalid='ignore') # silence maths warnings (0/0 or X/0) globally
# import os
import pandas as pd
from pathlib import Path
import rasterio as rio
import rasterio.merge
# from rasterio.enums import Resampling
from rasterio import features
# from rasterio.features import shapes
# from rasterio.features import sieve
from rasterio.merge import merge
# from rasterio.plot import show
from rasterio.windows import from_bounds
from rasterstats import zonal_stats
import rioxarray
import shapely
from shapely.geometry import shape
# from shapely.geometry import Polygon
# from shapely import affinity
import xarray as xr
from xrspatial import hillshade

#import cartopy.crs as ccrs
import matplotlib.pyplot as plt

#from cartopy.crs import PlateCarree


In [2]:
def get_bands_by_satellite(target_colours):
    records = []

    # Map colors to band numbers based on the Satellite ID prefix
    # LT04/05 = Landsat 4/5 TM; LE07 = Landsat 7 ETM+; LC08/09 = Landsat 8/9 OLI
    band_config = {
    "LT05":{"B1":"BLUE", "B2":"GREEN", "B3":"RED", "B4":"NIR", "B5":"SWIR1", "B7":"SWIR2"},
    "LT07":{"B1":"BLUE", "B2":"GREEN", "B3":"RED", "B4":"NIR", "B5":"SWIR1", "B6":"TIR", "B7":"SWIR2" },
    "LC08":{"B1":"COAST/AERO", "B2":"BLUE", "B3":"GREEN", "B4":"RED", "B5":"NIR", "B6":"SWIR1", "B7":"SWIR2"},
    "LC09":{"B1":"COAST/AERO", "B2":"BLUE", "B3":"GREEN", "B4":"RED", "B5":"NIR", "B6":"SWIR1", "B7":"SWIR2"}
}
    for folder in PATHS["landsat_images"].iterdir():
        if folder.is_dir():
            for file in folder.glob("*_SR_B*.TIF"):
                parts = file.name.split("_")
                sat_id = parts[0][:4]
                band_id = parts[-1].replace(".TIF", "")
                if band_config[sat_id][band_id] in target_colours:
                    records.append({
                        "satellite": sat_id,
                        "path_row": parts[2],
                        "year": parts[3][:4],
                        "band": band_id,
                        "colour": band_config[sat_id][band_id],
                        "filename": file.name,
                        "path": str(file)
                    })
    return pd.DataFrame(records)

In [3]:
def create_mosaic(file_list, out_path, dtype=None): 
    # 1. Get CRS from the first file for comparison with other files
    with rio.open(file_list[0]) as src:
        target_crs = src.crs
        out_meta = src.profile.copy()

    # 2. Compare othes files against ref CRS and reproject if they don't match
    processed_list = []
    for f in file_list:
        with rio.open(f) as src:
            if src.crs != target_crs:
                print(f"Fixing CRS for: {f.name}")
                processed_list.append(fix_projection(f, target_crs))
            else:
                processed_list.append(f)
                
    # 3. Merge images, return the new array and new spatial transform
    #print(f"    Mosaicking stared: {out_path.name}")
    mosaic, out_trans = merge(processed_list, nodata=0)

    # 4. Update the metadata with mosiac dimensions and transform
    bands, height, width = mosaic.shape
    out_meta.update({
        "height": height, 
        "width": width, 
        "transform": out_trans, 
        "nodata": 0,
        "dtype": dtype or out_meta['dtype'] # And here
    })

    # 5. Save mosaic and full metadata to file
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rio.open(out_path, "w", **out_meta) as dest:
        dest.write(mosaic)

    print(f"     Mosaic saved: {out_path.name}")

In [4]:
# def fix_projection(in_path, target_crs):
#     """Reprojects a file to the target CRS and returns the new path."""
#     out_path = in_path.parent / f"reprojected_{in_path.name}"
#     with rio.open('data_files/NI_Mosaic.tif') as src:
#     transform, width, height = rio.warp.calculate_default_transform(
#             src.crs, dst_crs, src.width, src.height, *src.bounds)
    
#     with rio.open(in_path) as src:
#         transform, width, height = calculate_default_transform(
#             src.crs, target_crs, src.width, src.height, *src.bounds)
#         meta = src.profile.copy()
#         meta.update({'crs': target_crs, 'transform': transform, 'width': width, 'height': height})

#         with rio.open(out_path, 'w', **meta) as dst:
#             reproject(source=rio.band(src, 1), destination=rio.band(dst, 1),
#                       src_transform=src.transform, src_crs=src.crs,
#                       dst_transform=transform, dst_crs=target_crs,
#                       resampling=Resampling.nearest)
#     return out_path

In [ ]:
# def reproject_raster(src, dst_crs):
#     transform, width, height = rio.warp.calculate_default_transform(
#         src.crs, dst_crs, src.width, src.height, *src.bounds
#     )

#     kwargs = src.meta.copy()
#     kwargs.update({
#         "crs": dst_crs,
#         "transform": transform,
#         "width": width,
#         "height": height
#     })

#     data = np.empty((src.count, height, width), dtype=src.dtypes[0])

#     for i in range(1, src.count + 1):
#         reproject(
#             source=rio.band(src, i),
#             destination=data[i-1],
#             src_transform=src.transform,
#             src_crs=src.crs,
#             dst_transform=transform,
#             dst_crs=dst_crs,
#             resampling=Resampling.nearest
#         )

#     return data, kwargs

In [5]:
def ndi_and_clipping(yr, border_gdf):
    """
    1. Loads windowed mosaic bands for the year.
    2. Stacks them into an Xarray Dataset.
    3. Clips the stack to the jagged park boundary.
    4. Calculates and saves each NDI index.
    """
    raster_data = {}
    win_meta = None
    
    for band in ANALYSIS_COLOURS:
        file = PATHS["mosaics"] / f"{yr}_{band}_mosaic.tif"
        with rio.open(file) as src:
            if border_gdf.crs != src.crs:
                border_gdf = border_gdf.to_crs(src.crs)
                
            window = from_bounds(*border_gdf.total_bounds, transform = src.transform)
            raster_data[band] = xr.DataArray(src.read(1,window=window).astype("float32"), dims=("y","x"))#flaot32 added here to support the  ndvi calulations later on
            if win_meta is None:
                win_meta = {'crs':src.crs, 'transform':src.window_transform(window)}

    ds = xr.Dataset(raster_data)
    ds = ds.rio.write_crs(win_meta['crs']).rio.write_transform(win_meta['transform'])

    ds_clipped = ds.rio.clip(border_gdf.geometry, drop =True)
            
    for NDI in ANALYSIS_TASKS:
        b1_name, b2_name = INDEX_TO_BANDS[NDI]
        b1, b2 = ds_clipped[b1_name], ds_clipped[b2_name]   

        result = (b1 - b2) / (b1 + b2)
        result = result.where(~np.isinf(result)) # Replaces Inf with NaN
        
        ndi_out = PATHS["ndi"] / f"{yr}_{NDI}.tif"
        result.rio.to_raster(ndi_out, nodata=np.nan)
        print(f"     Processed and saved: {yr}_{NDI}.tif")
    del result, ds_clipped, raster_data


In [6]:
# creating composite images
def normalize(band):
    #data_only = band[band>0]
    #vmin, vmax = np.nanpercentile(band, [1, 99])
    vmax, vmin = band.max(), band.min()
    norm = np.clip(band, vmin, vmax)
    norm = ((norm - vmin)/(vmax - vmin))
   #norm = np.power(norm, 3) 
#    norm[band == 0] = 0
    return norm
   
def composite_images(yr, disp_type, input_path, dest_path):
        path1 = input_path / f"{yr}_{INDEX_TO_BANDS[disp_type][0]}_mosaic.tif"
        path2 = input_path / f"{yr}_{INDEX_TO_BANDS[disp_type][1]}_mosaic.tif"
        path3 = input_path / f"{yr}_{INDEX_TO_BANDS[disp_type][2]}_mosaic.tif"
    
        with rio.open(path1) as src1, rio.open(path2) as src2, rio.open(path3) as src3:
            colr1 = src1.read(1)
            colr2 = src2.read(1)
            colr3 = src3.read(1)
            out_meta = src1.meta.copy()
            ncolr1 = (normalize(colr1) * 255).astype(np.uint8)
            ncolr2 = (normalize(colr2) * 255).astype(np.uint8)
            ncolr3 = (normalize(colr3) * 255).astype(np.uint8)
            
            out_meta.update({
                "driver": "GTiff",
                "count": 3,
                "dtype": "uint8",
                "compress": "lzw"
            })
            out_filename = dest_path / f"{yr}_{disp_type}_composite.tif"

            with rio.open(out_filename, 'w', **out_meta) as dst:
                dst.write(ncolr3, 1) # Red channel (or first selected band)
                dst.write(ncolr2, 2) # Green channel
                dst.write(ncolr1, 3) # Blue channel
                
            print(f"     Saved: {out_filename.name}")

In [ ]:
# --- 1. User Inputs ---
base = Path("C:/RS_GIS/EGM722/Assignment/Grand_Canyon/USGS_data/Unzipped") # insert path of base directory into brackets, eg. Path(<base_folder_location>)
boundary_file = Path("C:/RS_GIS/EGM722/Assignment/Grand_Canyon/USGS_data/Unzipped/Boundary Files/national_park_boundary.shp")

#select desired display and analysis: 0 = not wanted, 1 = wanted.
USER_OPTIONS = {
    "TRUE_COLOUR": 1,
    "FALSE_COLOUR": 0,
    "NDVI": 1,
    "NDWI": 1,
    "NDSI": 0,
}
print("1. User Inputs and selections accepted")

# --- 2. Creating folder structure ---
PATHS = {
    "landsat_images" : base / "Landsat_Images",
    "ndi": base / "NDI Images",
    "mosaics": base / "Mosaics",
    "boundaries": base / "Boundary Files",
    "earthaccess": base / "EarthAccess",
    "results": base / "Results"
}
for p in PATHS.values(): p.mkdir(parents=True, exist_ok=True)
print("2. Directories checked/created")

# --- 3. Acquiring DEMs ---
border_poly_proj = border_gdf.to_crs(epsg = 4326).union_all() #projection EPSG 4326 needed for earthacess
search_area = border_poly_proj.minimum_rotated_rectangle
search_area = shapely.geometry.polygon.orient(search_area, sign=1)
#getting dem data from earthaccess
print("7. EarthAccess data acquistion started...")
earthaccess.login(strategy='netrc')
earthaccess_files = earthaccess.search_data(
    short_name = 'ASTGTM',
    polygon = search_area.exterior.coords
)
if len(earthaccess_files) == 0:
    print("  No Data Found in Earth Access")
    #code here to stop whole program?
    
earthaccess_dest = PATHS["earthaccess"]
downloaded_files = earthaccess.download(earthaccess_files, earthaccess_dest)


# --- 4. create hillshde
hillshade_orig = hillshade(dem_ds, azimuth=225, angle_altitude=45)
hillshade_26912 = hillshade_orig.rio.reproject("EPSG:26912")

# Pull the variable out, fill it, and cast it into its own separate object
hill_final = (hillshade_26912.elevation.fillna(0) * 255).astype("uint8")

# Set the nodata value explicitly on this new object so it saves correctly
hill_final.rio.write_nodata(0, inplace=True)

hillshade_dest = PATHS["results"] / "hillshade.tif"
# Use 'hill_final' here, NOT 'hillshade_26912'
hill_final.rio.to_raster(hillshade_dest, compress="lzw", dtype="uint8")




# --- 5. Determine colour bands requried for selected disaply / analysis 
#dictionary describes relationship between actions and bands
INDEX_TO_BANDS = {
    "TRUE_COLOUR" : ["BLUE", "GREEN", "RED"],
    "FALSE_COLOUR" : ["GREEN","RED","NIR"],
    "NDVI": ["RED", "NIR"],
    "NDWI": ["GREEN", "NIR"],
    "NDSI": ["GREEN", "SWIR1"]
}

DISPLAY_TASKS = [task for task in ["TRUE_COLOUR", "FALSE_COLOUR"] if USER_OPTIONS.get(task)]
display_colours_raw = list({band for key in DISPLAY_TASKS for band in INDEX_TO_BANDS[key]})
colour_order = ["COAST/AERO","BLUE", "GREEN", "RED", "NIR", "SWIR1", "SWIR2"]
DISPLAY_COLOURS = [b for b in colour_order if b in display_colours_raw]
ANALYSIS_TASKS = [task for task in ["NDVI", "NDWI", "NDSI"] if USER_OPTIONS.get(task)]
ANALYSIS_COLOURS = list({band for key in ANALYSIS_TASKS for band in INDEX_TO_BANDS[key]})
ALL_COLOURS = list(set(DISPLAY_COLOURS + ANALYSIS_COLOURS))
print("3. Band identification and mapping completed")




# --- 6. Build DataFrame of raster images needed for display / analysis activities ---
df = get_bands_by_satellite(ALL_COLOURS)
df.to_csv(PATHS["landsat_images"] / "bands_dataframe.csv", index = False)
print("4. Lansdat image dataset created")


# --- 7. Mosaicking rasters---
years = df["year"].unique()
print("5. Ratser mosaicking started...")
for yr in years:
    for colr in ALL_COLOURS:
        file_list = df[(df["year"] == yr) & (df["colour"] == colr)]["path"].tolist()
        dst_path = PATHS["mosaics"] / f"{yr}_{colr}_mosaic.tif"
        create_mosaic(file_list, dst_path)
print("     All rasters mosaicked")

# --- 8. Mosaicking DEM---
print("8. DEM mosaicking started...")
dem_files = [fn for fn in downloaded_files if 'dem.tif' in fn.name]
dem_mosaic_dest = PATHS["earthaccess"] / "DEM_mosaic.TIF"
dem_meta = create_mosaic(dem_files, dem_mosaic_dest, dtype='float32')


# --- 9. NDVI analysis and image clipping---
print("6. Normalised difference analysis and clipping started...")
border_gdf = gpd.read_file(boundary_file)
for yr in years:
    ndi_and_clipping(yr, border_gdf)
print("     Calculations and clipping completed")

with rio.open(dem_mosaic_dest) as src:
    border_gdf_proj = border_gdf.to_crs(src.crs)
    
    window = from_bounds(*border_gdf_proj.total_bounds, transform = src.transform)
    dem_windowed = xr.DataArray(src.read(1,window=window).astype("float32"), dims=("y","x"))
    win_meta = {'crs':src.crs, 'transform':src.window_transform(window)}

dem_ds = xr.Dataset({"elevation":dem_windowed})
dem_ds = dem_ds.rio.write_crs(win_meta['crs']).rio.write_transform(win_meta['transform'])
dem_ds_clipped = dem_ds.rio.clip(border_gdf_proj.geometry, drop =True)
print("9. DEM clipped to park boundary")


# --- 10. Vectorising clipped DEM ---
print("10. DEM Vectorisation started...")
step = 250
dem_ds_classified = (dem_ds_clipped.elevation.values // step).astype("int32")
mask = ~np.isnan(dem_ds_clipped.elevation.values)
shapes = features.shapes(dem_ds_classified, mask=mask,transform=dem_ds_clipped.rio.transform())
polygons = []
for i, (geom, value) in enumerate(shapes):
    zone_label = int(value*step)
    polygons.append({
        'poly_id':i,
        'elevation_zone': zone_label,
        'geometry':shape(geom)
    })

segments_gdf = gpd.GeoDataFrame(polygons, crs=dem_ds_clipped.rio.crs)
print("      Vectorising completed")


# --- 11. Caclualting Zonal Statistics ---
print("11. Calculating Zonal Statistics...")
ndvi_path = PATHS['ndi']/"2020_NDVI.tif" # path for NDVI image to be cmopared against segmented DEM
stats = zonal_stats(segments_gdf,ndvi_path,stats='mean') # Runing zonal statistics on ndvi_path
segments_gdf['Mean NDVI'] = [s['mean'] for s in stats] # Attaching results to GeoDataFrame
segments_gdf = segments_gdf.dropna(subset = 'Mean NDVI') # cleaning up empty or NoData zones
segments_gdf.to_csv(PATHS['results']/"zonal_statistics.csv", index=False)
print("      Calculated")

# --- 12. Creating images for dispaly ---
print("12. Creating display images...")
for yr in years:
    for disp in DISPLAY_TASKS:
        composite_images(yr, disp, PATHS["mosaics"], PATHS["results"])


print("      ALL images created ")

print("--- All Tasks Completed ---")

1. User Inputs and selections accepted
2. Directories checked/created
3. Band identification and mapping completed
4. Lansdat image dataset created
5. Ratser mosaicking started...
     Mosaic saved: 2020_NIR_mosaic.tif
     Mosaic saved: 2020_GREEN_mosaic.tif
     Mosaic saved: 2020_RED_mosaic.tif
     Mosaic saved: 2020_BLUE_mosaic.tif
     All rasters mosaicked
6. Normalised difference analysis and clipping started...
     Processed and saved: 2020_NDVI.tif
     Processed and saved: 2020_NDWI.tif
     Calculations and clipping completed
7. EarthAccess data acquistion started...


QUEUEING TASKS | :   0%|          | 0/16 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/16 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/16 [00:00<?, ?it/s]

8. DEM mosaicking started...
     Mosaic saved: DEM_mosaic.TIF
9. DEM clipped to park boundary
10. DEM Vectorisation started...


In [ ]:
#Plotting results

comp_path = PATHS["results"] / "2020_TRUE_COLOUR_composite.tif"
hill_path = PATHS["results"] / "hillshade.tif"
boundary = gpd.read_file(boundary_file)

with rio.open(comp_path) as src:
    img_extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
    rgb_composite = src.read([1,2,3]).transpose(1,2,0)/255
    
boundary = boundary.to_crs(src.crs)
gdf_results = segments_gdf.dropna(subset=['Mean NDVI']) # cleaning up empty or NoData zones
gdf_results = gdf_results.to_crs(src.crs)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(rgb_composite, extent=img_extent)
ax.imshow(hillshade, cmap='gray', alpha=0.4, extent=img_extent)
boundary.plot(ax=ax, facecolor='none', edgecolor='orange', linecolor='orange', linewidth=1)
gdf_results.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=0.5)

ax.set_axis_off()
plt.show()



#     fig.colorbar(ax.get_images()[0], ax=ax, label='NDVI Value')

In [ ]:
import matplotlib.pyplot as plt
import rioxarray

# Pick one of your generated files
#test_file = PATHS["ndi"] / "2020_NDWI.tif"
tru_colr_file=PATHS["results"]/"2020_TRUE_COLOUR_composite.tif"
hillshade_file=PATHS["results"]/"hillshade.tif"

with rio.open(tru_colr_file) as src:
    rds_extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
    rds = src.read([1,2,3]).transpose(1,2,0).astype("float32")/255

with rio.open(hillshade_file) as src2:
    rds2 = src2.read(1).astype("float32")/255
    
boundary = gpd.read_file(boundary_file)
boundary = boundary.to_crs(src.crs) 
# Open it
#rds = rioxarray.open_rasterio(tru_colr_file)
#rds2 = rioxarray.open_rasterio(hillshade_file)

# Plot it
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(rds, extent=rds_extent)
ax.imshow(rds2, cmap='gray', alpha=0.4, extent=rds_extent)
boundary.plot(ax=ax, facecolor='none', edgecolor='orange', linewidth=1)
#gdf_results.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=0.5)

# plt.figure(figsize=(10, 20))
# rds.plot(,robust=True)#, vmin=-1, vmax=1) # Standard NDVI colors
# rds2.plot(cmap=grey,robust=True)
# plt.title(f"Check: {test_file.name}")
# plt.show()

In [ ]:
import rasterio as rio
import numpy as np
# import cv2 # Make sure you ran 'pip install opencv-python'


# base = Path("C:/RS_GIS/EGM722/Assignment/Grand_Canyon/USGS_data/Unzipped")
# PATHS = {
#     "landsat_images" : base / "Landsat_Images",
#     "ndi": base / "NDI Images",
#     "mosaics": base / "Mosaics",
#     "boundaries": base / "Boundary Files",
#     "earthaccess": base / "EarthAccess",
#     "results": base / "Results"
# }
# print("12. Creating display images...")
# years = df["year"].unique()
# for yr in years:
#     for disp in DISPLAY_TASKS:
#         composite_images(yr, disp, PATHS["mosaics"], PATHS["results"])
        
scale = 4

img = rioxarray.open_rasterio(PATHS["results"]/"2020_TRUE_COLOUR_composite.tif")
img_height_new = int(img.rio.height / scale)
img_width_new = int(img.rio.width / scale)
img_scaled = img.rio.reproject('EPSG:4326', shape=(img_height_new,img_width_new))
b = img_scaled.rio.bounds()
img_bounds_folium = [[b[1],b[0]],[b[3],b[2]]]
img_data = img_scaled.transpose('y','x','band').values
if img_data.max() <= 1.0:
    img_data = (img_data * 255).astype('uint8')
alpha = np.where(img_data.sum(axis=-1) > 0, 255, 0).astype('uint8')
img_data_rgba = np.dstack((img_data, alpha))



# hill = rioxarray.open_rasterio(PATHS["results"]/"hillshade.tif")
# hill_height_new = int(hill.rio.width / scale)
# hill_width_new = int(hill.rio.width / scale)
# hill_scaled = hill.rio.reproject('EPSG:4326', shape=(hill_height_new,hill_width_new))
# h = hill_scaled.rio.bounds()
# hill_bounds_folium = [[h[1],h[0]],[h[3],h[2]]]
# hill_data_2d = hill_scaled.values.squeeze()
# #beta = np.where(hill_data.sum(axis=-1) > 0, 255, 0).astype('uint8')
# #hill_data_2d = np.dstack((hill_data, beta))

m = folium.Map([36.15, -112.7],tiles="USGS.USImagery")
folium.raster_layers.ImageOverlay(
    name = "True Colour Overaly",
    image = img_data_rgba,
    bounds = img_bounds_folium,
).add_to(m)
# folium.raster_layers.ImageOverlay(
#     name = "Hillshade Overaly",
#     image = hill_data_2d,
#     bounds = hill_bounds_folium,
#     opacity = 0.4    
# ).add_to(m)


folium.LayerControl().add_to(m)
m

In [ ]:
# def norm(band):
#     band_min, band_max = band.min(), band.max()
#     return ((band - band_min)/(band_max - band_min))

# for i in range(len(b2_file)):   
    
#     # Open each band using gdal
#     b2_link = gdal.Open(b2_file[i])
#     b3_link = gdal.Open(b3_file[i])
#     b4_link = gdal.Open(b4_file[i])
    
#     # call the norm function on each band as array converted to float
#     b2 = norm(b2_link.ReadAsArray().astype(np.float))
#     b3 = norm(b3_link.ReadAsArray().astype(np.float))
#     b4 = norm(b4_link.ReadAsArray().astype(np.float))
    
#     # Create RGB
#     rgb = np.dstack((b4,b3,b2))
#     del b2, b3, b4
    
#     # Visualize RGB
#     #import matplotlib.pyplot as plt
#     #plt.imshow(rgb)
    
#     # Export RGB as TIFF file
#     # Important: Here is where you can set the custom stretch
#     # I use min as 2nd percentile and max as 98th percentile
#     sm.toimage(rgb,cmin=np.percentile(rgb,2),
#                cmax=np.percentile(rgb,98)).save(b2_file[i].split('_01_')[0]+'_RGB.tif')
    

In [ ]:
# # --- 12. Creating images for dispaly ---
# print("12. Creating display images...")
# for yr in years:
#     for disp in DISPLAY_TASKS:
#         composite_images(yr, disp, PATHS["mosaics"], PATHS["results"])


In [ ]:
import matplotlib.pyplot as plt

img_file = PATHS["results"]/"2020_TRUE_COLOUR_composite.tif"

rds = rio.open(img_file)
# # 1. Create the plot
# plt.figure(figsize=(12, 8))

# # 2. Plot the elevation layer
# # robust=True ignores the extreme outliers (like NoData edges) to stretch the colors over the actual land
# ds_clipped.elevation.plot(cmap='terrain', robust=True)

# # 3. Add titles and labels
# plt.title(f"Clipped DEM: {boundary_file.stem}")
# plt.xlabel("Longitude" if ds_clipped.rio.crs.is_geographic else "Easting (m)")
# plt.ylabel("Latitude" if ds_clipped.rio.crs.is_geographic else "Northing (m)")

# plt.show()

# # 4. Quick stats check to confirm values are realistic
# print(f"Min Elevation: {ds_clipped.elevation.min().values:.2f} m")
# print(f"Max Elevation: {ds_clipped.elevation.max().values:.2f} m")

In [ ]:
# Plots the mosaicked DEM layer
#data_crs=ccrs.PlateCarree()
dem = rioxarray.open_rasterio(dem_mosaic_dest)

dem.plot(cmap='terrain', robust=True) 
plt.title("DEM Sanity Check")
plt.show()


# fig, ax = plt.subplots(1, 1, subplot_kw=dict(projection=ccrs.PlateCarree()))
# ax.imshow(dem, cmap='gray', transform=ccrs.PlateCarree(), extent=[xmin, xmax, ymin, ymax]) # display band 0 as a grayscale image
# ax.set_extent([xmin, xmax, ymin, ymax], crs=ccrs.PlateCarree())
# plt.show()

In [ ]:

comp_path = PATHS["results"] / "2020_TRUE_COLOUR_composite.tif"
hill_path = PATHS["results"] / "hillshade.tif"
boundary = gpd.read_file(boundary_file)

with rio.open(comp_path) as src:
    img_extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
    rgb_composite = src.read([1,2,3]).transpose(1,2,0)/255.dtype="float32"
    
boundary = boundary.to_crs(src.crs)
#gdf_results = gdf_results.to_crs(src.crs)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(rgb_composite, extent=img_extent)
ax.imshow(hillshade, cmap='gray', alpha=0.4, extent=img_extent)
boundary.plot(ax=ax, facecolor='none', edgecolor='orange', linewidth=1)
#gdf_results.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=0.5)

ax.set_axis_off()
plt.show()

In [ ]:
def get_buffered_window(gdf, src, padding):
    """
    gdf: The park boundary
    src: The open mosaic file
    padding: The distance to expand the box (e.g., 5000 for metres, 0.05 for degrees)
    """
    left, bottom, right, top = gdf.total_bounds
    
    # Expand the box
    buffered_bounds = (left - padding, bottom - padding, right + padding, top + padding)
    
    # Create the window
    win = from_bounds(*buffered_bounds, transform=src.transform)
    return win

# --- TO TEST DIFFERENT SIZES ---
# If your CRS is 4326 (DEM), try padding=0.05 or 0.1
# If your CRS is 26912 (UTM), try padding=5000 or 10000


# Use a large number for padding because your composite is in Metres (UTM)
padding_meters = 10000 

    
with rio.open(PATHS["results"] / "2020_TRUE_COLOUR_composite.tif") as src:
    proj_gdf = border_gdf.to_crs(src.crs)
    display_window = get_buffered_window(proj_gdf, src, padding=padding_meters)
    
    # This line translates pixel math to the real-world coordinates for the plot
    from rasterio.plot import plotting_extent
    win_extent = plotting_extent(src.read(1, window=display_window), src.window_transform(display_window))
    
    test_data = src.read(window=display_window)
    from rasterio.plot import reshape_as_image
    
    # Add 'extent' here so the image knows WHERE it is
    plt.imshow(reshape_as_image(test_data), extent=win_extent)
    
    # Now this line will actually line up
    proj_gdf.plot(ax=plt.gca(), facecolor="none", edgecolor="red")
    plt.show()

    
    


In [ ]:
# adding buffer to park boundary to avoid clipping river only
all_polygons = []
park_poly_file = "C:/RS_GIS/EGM722/Assignment/Grand_Canyon/shaprfiles/national_park_boundary.shp"
park_poly = gpd.read_file(park_poly_file).to_crs(epsg = 26912).union_all()
park_ploy_buffered = park_poly.buffer(1000)
park_buff_gdf = gpd.GeoDataFrame(geometry=[park_ploy_buffered],crs=('EPSG:26912'))
park_ref_out = polygons / "buffered_park_polygon.gpkg"
park_buff_gdf.to_file(park_ref_out, driver ="GPKG")

In [ ]:
#function for isolationg and joing the river polygons
def merging_polygons(poly_gdf, ref_poly):
    combined_gdf = pd.concat(poly_gdf)
    
    #Apply buffer to bridge gaps between tiles
    combined_gdf['geometry'] = combined_gdf.geometry.buffer(100) 
    
    #Weld small localised pieces together
    merged = combined_gdf.union_all()
    
    # Searate main bodies into distinct polygons
    main_polygons = gpd.GeoSeries([merged], crs="EPSG:26912").explode(index_parts=False)
    main_polygons_gdf = gpd.GeoDataFrame(geometry=main_polygons)
   
    # Keep only polygons that touch river reference
    # ensures small 'ilsnad' polyons in the river path are not omitted.
    ref_geom = gpd.read_file(ref_poly).to_crs("EPSG:26912").union_all()
    referenced_poly = main_polygons_gdf[main_polygons_gdf.intersects(ref_geom)]
   
    # 5. Shrink it back (undo the buffer) to get original river width
    referenced_poly['geometry'] = referenced_poly.geometry.buffer(-100)
    buffered_poly = referenced_poly.copy()
    buffered_poly['geometry'] = buffered_poly.geometry.buffer(5000)

    poly_out = polygon_data / "Grand_Canyon_River.gpkg"
    buff_out = polygon_data / "Grand_Canyon_River_Buffer_5000.gpkg"
    referenced_poly.to_file(poly_out, driver="GPKG")
    buffered_poly.to_file(buff_out, driver="GPKG")

In [ ]:
### function for making mask, binarising it, and converting to polygons
def mask_from_raster(year, pathrow, NDI):#, river_ref_path):
    poly_list = []
    file = NDI_data/f"{year}_{pathrow}_{NDI}.tif"
    with rio.open(file) as src:
        non_binary = src.read(1)
        mask = np.where(non_binary > -0.1 , 255 ,0).astype("uint8")
        cleaned_mask = sieve(mask,50)
        out_meta = src.meta.copy()
        out_meta.update(dtype = "uint8", nodata = 0)
        mask_out = masks / f"{year}_{pathrow}_{NDI}.tif"
        
        with rio.open(mask_out, "w", **out_meta) as dst:
            dst.write(cleaned_mask,1)

    file = mask_data / f"{year}_{pathrow}_{NDI}.tif"
    with rio.open(file) as src:
        tile = src.read(1).astype("int16")
        out_meta = src.meta.copy()
        mask_shapes = features.shapes(
            tile,
            mask = tile > 0,
            transform = out_meta['transform']
        )
    # The corrected list comprehension
    polygons = [
    {'geometry':shape(s),'properties':{'id': i}}
     for i, (s,v) in enumerate(mask_shapes)
    ]
              
    gdf = gpd.GeoDataFrame.from_features(polygons, crs=out_meta['crs'])
    return gdf